## ISR(Import Shift Ratio): 수입 흐름 전이 지표

ISR(Import Shift Ratio)은 특정 품목에 대해 **규제 대상국 A로부터의 수입 감소가 제3국 B로부터의 수입 증가로 전이되었는지**를 측정하기 위한 지표이다. 즉, 반덤핑 관세나 수입 규제 이후 한국이 규제 대상국 A로부터 수입하던 물량이 감소하는 동시에, 일정 시차를 두고 제3국 B로부터의 수입 물량이 증가하는 패턴이 나타나는지를 확인한다.

본 연구에서 ISR은 스피어만 순위상관계수(Spearman rank correlation)를 기반으로 정의한다. 특정 품목 $p$에 대해 한국의 규제 대상국 A로부터의 수입 물량 시계열을 $M^{(p)}_{A,t}$, 한국의 제3국 B로부터의 수입 물량 시계열을 $M^{(p)}_{B,t+l}$라고 할 때, ISR은 다음과 같이 계산된다.

$$
ISR^{(p,l)}_{A \to B}
=
1 -
\frac{
6 \sum_{t=1}^{n}
\left[
rank(M^{(p)}_{A,t}) - rank(M^{(p)}_{B,t+l})
\right]^2
}{
n(n^2 - 1)
}
$$

여기서 $l$은 우회무역이 발생하는 데 걸리는 시차(lag)를 의미한다. 무역 규제 이후 공급망이 즉시 전환될 수도 있지만, 실제로는 선적, 통관, 재수출, 계약 변경 등의 과정으로 인해 일정한 시간이 소요될 수 있다. 따라서 본 연구에서는 lag를 하나로 고정하지 않고, 다음 네 가지 경우를 모두 고려한다.

$$
l \in \{0, 1, 2, 3\}
$$

각 lag에 대해 ISR을 각각 계산한 뒤, 그중 가장 작은 값을 최종 ISR 지표로 사용한다.

$$
ISR^{(p)}_{A \to B}
=
\min_{l \in \{0,1,2,3\}}
ISR^{(p,l)}_{A \to B}
$$

ISR 값은 -1에서 1 사이의 값을 가지며, 해석은 다음과 같다.

- $ISR \approx -1$: 규제 대상국 A로부터의 수입이 감소할 때 제3국 B로부터의 수입이 증가하는 강한 역방향 관계가 존재한다. 즉, 우회무역 가능성이 높다.
- $ISR \approx 0$: 두 국가의 수입 흐름 사이에 뚜렷한 관계가 없다.
- $ISR \approx 1$: 두 국가의 수입 흐름이 같은 방향으로 움직인다. 이는 우회 전이라기보다 공통 수요 변화나 시장 동조화일 가능성이 크다.

따라서 본 연구에서는 lag 0\~3개월 중 **가장 강한 역방향 수입 전이 관계**를 포착하기 위해 최소 ISR 값을 대표값으로 정의한다. 이 방식은 우회무역이 즉각적으로 발생하는 경우뿐만 아니라, 1~3개월의 시차를 두고 나타나는 경우까지 함께 탐지할 수 있다는 장점이 있다.

In [102]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [84]:
sr_df = pd.read_csv(filepath_or_buffer = "sr_import_df.csv")
sr_df

,사건번호,국가,품목,hs_code,년월,수입량,수입총달러,단가,관세부과범위,부과시작일,부과종료일,규제국목록,규제국여부
0,0,남아프리카공화국,초산에틸(1차재심),291531,2012.03,0.0,0.0,NaN,3.14~14.17,2012-03-27,2015-03-26,"중국,싱가포르,일본",False
1,0,남아프리카공화국,초산에틸(1차재심),291531,2012.04,0.0,0.0,NaN,3.14~14.17,2012-03-27,2015-03-26,"중국,싱가포르,일본",False
2,0,남아프리카공화국,초산에틸(1차재심),291531,2012.05,0.0,0.0,NaN,3.14~14.17,2012-03-27,2015-03-26,"중국,싱가포르,일본",False
3,0,남아프리카공화국,초산에틸(1차재심),291531,2012.06,6.0,110.0,18.333333,3.14~14.17,2012-03-27,2015-03-26,"중국,싱가포르,일본",False
4,0,남아프리카공화국,초산에틸(1차재심),291531,2012.07,58.0,1790.0,30.862069,3.14~14.17,2012-03-27,2015-03-26,"중국,싱가포르,일본",False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20383,59,핀란드,탄소강과 그밖의 합금강 열간압연 후판제품,7208,2026.06,0.0,0.0,NaN,27.91~34.1,2025-11-24,2030-11-23,중국,False
20384,59,핀란드,탄소강과 그밖의 합금강 열간압연 후판제품,7208,2026.07,0.0,0.0,NaN,27.91~34.1,2025-11-24,2030-11-23,중국,False
20385,59,핀란드,탄소강과 그밖의 합금강 열간압연 후판제품,7208,2026.08,0.0,0.0,NaN,27.91~34.1,2025-11-24,2030-11-23,중국,False
20386,59,핀란드,탄소강과 그밖의 합금강 열간압연 후판제품,7208,2026.09,0.0,0.0,NaN,27.91~34.1,2025-11-24,2030-11-23,중국,False


In [94]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

def calculate_sr_with_lags(
    sr_df,
    window=6,
    lags=[0, 1, 2, 3],
    min_total_import=0
):
    """
    사건번호-품목-hs_code별로 규제국 A와 비규제국 B의 수입량 시계열을 비교하여
    lag별 Spearman 상관계수 기반 SR을 계산한다.
    """

    df = sr_df.copy()

    # 날짜 정리
    df = df[df["년월"].astype(str).str.match(r"^\d{4}\.\d{2}$")].copy()
    df["년월_dt"] = pd.to_datetime(df["년월"].astype(str), format="%Y.%m")

    # 숫자 정리
    df["수입량"] = pd.to_numeric(df["수입량"], errors="coerce").fillna(0)

    results = []

    group_cols = ["사건번호", "품목", "hs_code"]

    for (case_id, product_name, hs_code), g in df.groupby(group_cols):
        g = g.copy()

        # 분석 시작월: 해당 사건-품목의 가장 이른 월
        t0 = g["년월_dt"].min()

        # lag 3까지 고려하기 위해 6+3개월 필요
        full_months = pd.date_range(
            start=t0,
            periods=window + max(lags),
            freq="MS"
        )

        # 월-국가별 수입량 pivot
        pivot = (
            g.pivot_table(
                index="년월_dt",
                columns="국가",
                values="수입량",
                aggfunc="sum"
            )
            .reindex(full_months)
            .fillna(0)
        )

        # 규제국 목록
        regulated_countries = (
            g.loc[g["규제국여부"] == True, "국가"]
            .dropna()
            .unique()
            .tolist()
        )

        # 비규제 후보국 목록
        candidate_countries = (
            g.loc[g["규제국여부"] == False, "국가"]
            .dropna()
            .unique()
            .tolist()
        )

        for A in regulated_countries:
            if A not in pivot.columns:
                continue

            A_full = pivot[A]

            # 규제국 A의 6개월 시계열
            A_series = A_full.iloc[:window].reset_index(drop=True)

            # 규제국도 6개월 내내 변화 없으면 Spearman 불가
            if A_series.nunique() <= 1:
                continue

            for B in candidate_countries:
                if B not in pivot.columns:
                    continue

                B_full = pivot[B]

                # 후보국 B가 전체 필요 기간 동안 수입이 전혀 없으면 제외
                if B_full.sum() <= min_total_import:
                    continue

                sr_by_lag = {}

                for lag in lags:
                    B_series = B_full.iloc[lag:lag + window].reset_index(drop=True)

                    if len(B_series) < window:
                        sr_by_lag[f"sr_lag{lag}"] = np.nan
                        continue

                    # 후보국이 해당 lag window에서 상수면 계산 불가
                    if B_series.nunique() <= 1:
                        sr_by_lag[f"sr_lag{lag}"] = np.nan
                        continue

                    sr_value, _ = spearmanr(A_series, B_series)
                    sr_by_lag[f"sr_lag{lag}"] = sr_value

                valid_srs = {
                    lag: sr_by_lag.get(f"sr_lag{lag}")
                    for lag in lags
                    if pd.notna(sr_by_lag.get(f"sr_lag{lag}"))
                }

                if len(valid_srs) == 0:
                    sr_min = np.nan
                    sr_best_lag = np.nan
                    sr_risk = np.nan
                else:
                    sr_best_lag = min(valid_srs, key=valid_srs.get)
                    sr_min = valid_srs[sr_best_lag]
                    sr_risk = max(0, -sr_min)

                result = {
                    "사건번호": case_id,
                    "품목": product_name,
                    "hs_code": hs_code,
                    "규제국": A,
                    "후보국": B,
                    "분석시작월": t0.strftime("%Y.%m"),
                    "window": window,
                    **sr_by_lag,
                    "sr_min": sr_min,
                    "sr_best_lag": sr_best_lag,
                    "sr_risk": sr_risk,
                    "규제국_6개월_수입합": A_series.sum(),
                    "후보국_전체기간_수입합": B_full.sum()
                }

                results.append(result)

    return pd.DataFrame(results)

In [96]:
sr_result_df = calculate_sr_with_lags(
    sr_df,
    window=6,
    lags=[0, 1, 2, 3],
    min_total_import=0
)

sr_result_df.head()

,사건번호,품목,hs_code,규제국,후보국,분석시작월,window,sr_lag0,sr_lag1,sr_lag2,sr_lag3,sr_min,sr_best_lag,sr_risk,규제국_6개월_수입합,후보국_전체기간_수입합
0,0,초산에틸(1차재심),291531,일본,남아프리카공화국,2012.03,6,0.030359,-0.231908,-0.202920,-0.085714,-0.231908,1,0.231908,1277.0,215.0
1,0,초산에틸(1차재심),291531,일본,독일,2012.03,6,-0.085714,0.086966,-0.202920,-0.405840,-0.405840,3,0.405840,1277.0,571.0
2,0,초산에틸(1차재심),291531,일본,미국,2012.03,6,0.173931,-0.840668,0.231908,0.819689,-0.840668,1,0.840668,1277.0,1533.0
3,0,초산에틸(1차재심),291531,일본,스웨덴,2012.03,6,-0.676123,0.304256,0.654654,NaN,-0.676123,0,0.676123,1277.0,29.0
4,0,초산에틸(1차재심),291531,일본,스페인,2012.03,6,-0.101419,-0.304256,0.270449,-0.169031,-0.304256,1,0.304256,1277.0,2337.0


In [104]:
sr_result_df

,사건번호,품목,hs_code,규제국,후보국,분석시작월,window,sr_lag0,sr_lag1,sr_lag2,sr_lag3,sr_min,sr_best_lag,sr_risk,규제국_6개월_수입합,후보국_전체기간_수입합
0,0,초산에틸(1차재심),291531,일본,남아프리카공화국,2012.03,6,0.030359,-0.231908,-0.202920,-0.085714,-0.231908,1,0.231908,1277.0,2.150000e+02
1,0,초산에틸(1차재심),291531,일본,독일,2012.03,6,-0.085714,0.086966,-0.202920,-0.405840,-0.405840,3,0.405840,1277.0,5.710000e+02
2,0,초산에틸(1차재심),291531,일본,미국,2012.03,6,0.173931,-0.840668,0.231908,0.819689,-0.840668,1,0.840668,1277.0,1.533000e+03
3,0,초산에틸(1차재심),291531,일본,스웨덴,2012.03,6,-0.676123,0.304256,0.654654,NaN,-0.676123,0,0.676123,1277.0,2.900000e+01
4,0,초산에틸(1차재심),291531,일본,스페인,2012.03,6,-0.101419,-0.304256,0.270449,-0.169031,-0.304256,1,0.304256,1277.0,2.337000e+03
5,0,초산에틸(1차재심),291531,일본,영국,2012.03,6,0.880406,-0.246885,-0.617213,0.493771,-0.617213,2,0.617213,1277.0,1.302000e+03
6,0,초산에틸(1차재심),291531,일본,이탈리아,2012.03,6,-0.392792,0.130931,0.392792,-0.828079,-0.828079,3,0.828079,1277.0,1.200000e+03
7,0,초산에틸(1차재심),291531,일본,인도,2012.03,6,-0.771429,0.542857,0.347863,-0.434828,-0.771429,0,0.771429,1277.0,5.577512e+06
8,0,초산에틸(1차재심),291531,일본,프랑스,2012.03,6,-0.130931,0.654654,NaN,NaN,-0.130931,0,0.130931,1277.0,1.020000e+02
9,0,초산에틸(1차재심),291531,중국,남아프리카공화국,2012.03,6,0.030359,-0.492805,0.521794,-0.600000,-0.600000,3,0.600000,39160140.0,2.150000e+02


### ISR의 한계

ISR은 규제 대상국 A로부터의 수입 감소와 제3국 B로부터의 수입 증가가 시간적으로 반대 방향으로 움직이는지를 포착하는 데 유용하다. 그러나 ISR은 우회무역의 가능성을 보여주는 **탐지 지표**일 뿐, 그 자체로 우회무역을 확정하는 증거는 아니다. 본 연구에서 ISR을 해석할 때 고려해야 할 한계는 다음과 같다.

첫째, ISR은 스피어만 순위상관계수 기반 지표이므로 **물량의 절대 규모를 직접 반영하지 않는다**. 예를 들어 규제 대상국 A의 수입 감소분이 매우 크고 후보국 B의 수입 증가분이 매우 작더라도, 두 시계열의 순위가 반대로 움직이면 ISR 값은 크게 음수로 나타날 수 있다. 따라서 ISR은 반드시 후보국의 수입 증가 규모나 규제국 감소분 대비 흡수율과 함께 해석할 필요가 있다.

둘째, ISR은 **시계열의 방향성 관계**를 측정할 뿐, 실제 물류 경로를 직접 확인하지는 않는다. 즉, 한국의 A국 수입 감소와 B국 수입 증가가 동시에 나타났다고 해서 A국 물량이 실제로 B국을 거쳐 한국으로 들어왔다고 단정할 수는 없다. 이를 보완하기 위해서는 규제국 A에서 후보국 B로의 수출 증가 여부, 즉 $A \to B$ 방향의 교역 흐름을 추가로 확인할 필요가 있다.

셋째, 본 연구에서는 규제 이후 6개월 window를 사용하므로 표본 수가 작다. 6개월 자료에서 계산한 순위상관계수는 일부 월의 급격한 변동이나 일시적 거래에 민감하게 반응할 수 있다. 따라서 $ISR \approx -1$에 가까운 값이 나오더라도, 단기적 이상치나 일회성 거래에 의해 발생한 결과일 가능성을 함께 고려해야 한다.

넷째, 제3국 B의 수입 증가가 반드시 우회무역 때문이라고 볼 수는 없다. 후보국 B의 수입 증가는 국내 수요 변화, 가격 경쟁력 변화, 환율 변동, 공급 계약 변경, 기존 거래처 대체 등 정상적인 시장 요인에 의해서도 발생할 수 있다. 따라서 ISR은 단독으로 사용하기보다 가격 변화, 수입 규모 변화, 후보국의 기존 수입 패턴 등 다른 지표와 결합하여 해석해야 한다.

결론적으로 ISR은 우회무역 가능성이 있는 국가 쌍을 선별하는 1차 탐지 지표로 활용된다. 다만 우회무역 여부를 보다 설득력 있게 판단하기 위해서는 ISR과 함께 수입 증가 규모, 감소분 흡수율, 가격 변화, 규제국에서 후보국으로의 수출 흐름 등을 종합적으로 검토해야 한다.

---

## ESR(Export Shift Ratio): 수출 경로 전이 검증 지표

ESR(Export Shift Ratio)은 ISR을 통해 선별된 우회 의심 경로에 대해, **규제 대상국 A의 대한국 수출 감소가 후보국 B로의 수출 증가와 연결되어 있는지**를 확인하기 위한 검증 지표이다. 즉, ISR이 한국 수입시장 기준으로 $A \to Korea$ 수입 감소와 $B \to Korea$ 수입 증가를 포착한다면, ESR은 규제국 A의 수출시장 기준으로 $A \to Korea$ 수출 감소와 $A \to B$ 수출 증가가 함께 나타나는지를 확인한다.

우회무역이 실제로 발생했다면 다음과 같은 흐름이 관찰될 수 있다.

$$
A \to Korea \downarrow
$$

$$
A \to B \uparrow
$$

$$
B \to Korea \uparrow
$$

본 연구에서는 먼저 ISR을 통해 한국 수입시장 내에서 우회 의심 후보국 B와 가장 강한 전이 시차인 $l^*$를 찾는다.

$$
l^*
=
\argmin_{l \in \{0,1,2,3\}}
ISR^{(p,l)}_{A \to B}
$$

이후 ESR은 ISR에서 선택된 후보국 B와 최적 lag $l^*$에 대해서만 계산한다. 따라서 ESR은 lag 0~3을 다시 탐색하는 지표가 아니라, ISR에서 발견된 전이 시차가 규제국의 수출 경로 변화에서도 확인되는지를 검증하는 지표이다.

특정 품목 $p$에 대해 규제국 A의 대한국 수출 물량 시계열을 $X^{(p)}_{A \to KOR,t}$, 규제국 A의 후보국 B 대상 수출 물량 시계열을 $X^{(p)}_{A \to B,t+l^*}$라고 할 때, ESR은 다음과 같이 정의한다.

$$
ESR^{(p)}_{A \to B}
=
1 -
\frac{
6 \sum_{t=1}^{n}
\left[
rank(X^{(p)}_{A \to KOR,t}) - rank(X^{(p)}_{A \to B,t+l^*})
\right]^2
}{
n(n^2 - 1)
}
$$

여기서 $l^*$는 ISR에서 가장 강한 역방향 수입 전이 관계가 나타난 lag이다. 즉, ISR에서 $l^*=2$로 나타났다면 ESR 역시 규제국 A의 대한국 수출 시계열과 2개월 뒤의 규제국 A의 후보국 B 대상 수출 시계열을 비교한다.

ESR 값은 -1에서 1 사이의 값을 가지며, 해석은 다음과 같다.

- $ESR \approx -1$: 규제국 A의 대한국 수출이 감소할 때, 동일한 시차 구조에서 A의 후보국 B 대상 수출이 증가한다. 즉, 규제국의 수출 경로가 후보국 B로 전환되었을 가능성이 높다.
- $ESR \approx 0$: 두 수출 흐름 사이에 뚜렷한 관계가 없다.
- $ESR \approx 1$: A의 대한국 수출과 A의 후보국 B 대상 수출이 같은 방향으로 움직인다. 이는 우회 전이라기보다 A국의 전체 수출 경기나 품목 수요 변화에 따른 동조화일 가능성이 크다.

따라서 본 연구에서 ISR은 **우회 의심 후보국과 전이 시차를 탐색하는 1차 지표**, ESR은 **ISR에서 도출된 후보국과 lag를 이용해 규제국의 수출 경로 변화가 실제로 동반되었는지 확인하는 2차 검증 지표**로 활용된다. ISR과 ESR이 모두 낮게 나타나는 경우, 한국 수입시장과 규제국 수출시장 양쪽에서 동일한 우회 경로 신호가 관찰되는 것이므로 해당 국가 쌍은 우회무역 가능성이 높은 경로로 해석할 수 있다.